# Does the model represent *illegality* as distinct from *harmfulness*?

**What this notebook does, in one paragraph.** We show a small open model 240 short scenarios arranged as a 2×2: illegal-and-harmful, illegal-but-harmless, legal-but-harmful, legal-and-harmless, four per topic so topics are matched. We record the model's internal state for each sentence and train the simplest possible detector, a linear probe, to read off *legal vs illegal*. The catch is that on the two "easy" corners legal is the same thing as not-harmful, so a probe trained there cannot tell legality from harm. So we train the legality probe **only on harmful sentences** and test it **only on harmless sentences from topics it never saw**, and the reverse; if it still works, legality is decodable independently of harm. We do the same for harm, measure the angle between the two directions, compare with just asking the model, and check what happens when legality words are removed.

**How to use it.** Run top to bottom. Every code cell has a note above it. Settings are in one cell (2). The clock starts at the ⏱ line; the dataset hand-check (cell 5) is the first clocked task and the one that decides whether the project is real.

**Where instructions live.** Step-by-step with checkboxes: the vault note *MATS 12 - Run Sheet (legality probe)*. The design in one page and the decisions that are yours: `journal/design-questions-legality-probe.md`. The hand-check columns: `data/SCENARIOS_COLUMNS.md`.

**Prior work this must answer in the first paragraph:** Sadhu et al. 2026 (arXiv 2608.16852), Schwarz 2026 (2607.13075); method precedents Baez et al. 2026 (2607.07003), Shah et al. 2025 (2507.21141), Boxo et al. 2025 (2509.21344). This project adds the legality × harm factorial with conditional generalisation across strata, the angle between factorial directions, and an intervention.

### 1 · GPU check
**Runtime → Change runtime type → GPU.** A free T4 is enough; this project is one forward pass per sentence.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "NO GPU — Runtime → Change runtime type → GPU")

### 2 · Settings — the only cell you edit
- `MODEL`: Qwen3.5-4B first; 9B if you have an L4/A100 and time.
- `RUN`: a name for this session's files (the prompted condition is saved as `<RUN>_prompted`).
- `VAL_TOPICS` / `TEST_TOPICS`: how many of the 60 topics are held out for choosing the layer and for the final score. Do not change after seeing results.
- `DRIVE_DIR`: where results persist on your Google Drive.

In [ ]:
MODEL       = "Qwen/Qwen3.5-4B"
RUN         = "lp_4b"
VAL_TOPICS  = 15
TEST_TOPICS = 15
DRIVE_DIR   = "/content/drive/MyDrive/mats12_runs"


### 3 · Get the code and connect Drive (uncounted)
Clones the private repo (needs `GITHUB_TOKEN` in Colab Secrets), installs packages, moves `data/`, `figures/`, `journal/` onto Drive.

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
from getpass import getpass
try:
    token = userdata.get("GITHUB_TOKEN")          # Colab Secrets panel (key icon) → GITHUB_TOKEN, "Notebook access" ON
except Exception:
    token = None
if not token:
    print("No GITHUB_TOKEN found in Colab Secrets. The repo is private, so paste a GitHub fine-grained token")
    print("(GitHub → Settings → Developer settings → Fine-grained tokens; repository: mats12; Contents: read and write).")
    token = getpass("GitHub token: ").strip()
# The token travels in a per-command header, never in the saved remote URL (so it is not written to .git/config).
import base64
AUTH = ["-c", "http.extraheader=AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
os.environ["MATS12_GIT_AUTH"] = AUTH[1]
if not os.path.exists("/content/mats12"):
    r = subprocess.run(["git", *AUTH, "clone", "-q", "https://github.com/martinherje/mats12.git", "/content/mats12"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", *AUTH, "-C", "/content/mats12", "pull", "-q"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("git failed: " + r.stderr.replace(token, "<token>").strip() + "\nCheck the token has access to martinherje/mats12 (Contents: read).")
%cd /content/mats12
!git log --oneline -1
!bash scripts/colab_setup.sh "$DRIVE_DIR" 

### 4 · Load the model once (uncounted)
Expect 32 layers, width 2560, memory well under the card's total.

In [ ]:
!python scripts/gpu_smoke.py --model $MODEL

## ⏱ The clock starts here
Answer `journal/design-questions-legality-probe.md` (Files panel → `mats12/journal`), copy the answers into `journal/highlights.md` with the date, start Toggl.

### 5 · The dataset, and the hand-check that decides whether the project is real
`data/scenarios.csv` holds 240 candidate sentences: 60 topics × 4 quadrants, US law, written by Claude on 9 Sep and **not yet checked by anyone**. This cell validates the file and shows a sample. Your job, on the clock: open the CSV in the Files panel (double-click), read every sentence (off-diagonal quadrants first: `illegal_harmless`, `legal_harmful`, they are the test set), and for each row set `hand_checked=1`; fix `legal` / `harmful` where you disagree and set `relabelled=1` (and fix `quadrant` to match); set `borderline=1` where a competent lawyer could argue either way on the facts given; set `exclude=1` where the label depends on facts the sentence does not give. Save, re-run this cell until it says OK. Column meanings: `data/SCENARIOS_COLUMNS.md`. The write-up reports how many you read, relabelled and excluded.

In [ ]:
import pandas as pd
!python scripts/validate_scenarios.py data/scenarios.csv
df = pd.read_csv("data/scenarios.csv")
pd.set_option("display.max_colwidth", 120)
display(df[["id", "quadrant", "topic", "borderline", "text"]].sample(12, random_state=0))
print("\nEdit data/scenarios.csv in the Files panel (double-click), then re-run this cell to validate.")

### 6 · Record the model's internal state, in two conditions
**A. Bare sentence:** one forward pass per scenario, the state at the last token. **B. Prompted:** the same sentence inside the question "Is this illegal? Yes or No", the state at the token just before the model answers. Condition B is where the model is actually computing legality; A is what a passive monitor would see. Both are needed, because a probe that works only in B says "the model can judge legality when asked", and one that works in A says "it represents it unprompted". Look for "acts (240, 33, 2560)" twice.

In [ ]:
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios.csv --run $RUN --batch-size 16
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios.csv --run {RUN}_prompted --batch-size 16 --template chat --generation-prompt --enable-thinking off --instruction "Under the law of the United States, is the following action illegal? Answer with exactly one word: Yes or No.\n\nAction:" 

### 7 · The test that can actually answer the question: conditional generalisation, held-out topics
Training on the easy corners cannot separate "legality" from "not-harm": on those rows they are the same label. So the probe for *legal* is trained **inside one harm stratum** (legal-harmful vs illegal-harmful) and tested **on the other stratum** (legal-harmless vs illegal-harmless), on **topics it never saw**. Then the reverse. Then the symmetric pair for *harmful*. Layer and regularisation are chosen on validation topics only; the test topics are scored once; the interval is a topic-block bootstrap; and a permutation null repeats the whole selection on shuffled labels so it cannot inflate the result.

The same script estimates the two *factorial* directions on training topics (each contrast taken within a stratum of the other factor), reports the angle between them, and scores each on held-out topics — including the legality direction with the harm component removed.

Runs eight times: four designs × two conditions (bare, prompted). Read the four "TEST cross-stratum acc" lines per condition and the cosine.

In [ ]:
DESIGNS = [("legal", "harmful", "L_h2nh"), ("legal", "harmless", "L_nh2h"), ("harmful", "illegal", "H_i2l"), ("harmful", "legal", "H_l2i")]
for cond in [RUN, f"{RUN}_prompted"]:
    for target, stratum, tag in DESIGNS:
        print(f"\n################ {cond} · {tag}: {target} probe trained inside {stratum}, tested on the other stratum ################")
        !python scripts/probe_eval.py --run {cond} --target {target} --train-stratum {stratum} --tag {tag} --val-topics $VAL_TOPICS --test-topics $TEST_TOPICS

### 8 · The summary table, and the no-legality-words rerun
Collects the eight headline numbers. Then reruns the main design with every row containing explicit legality vocabulary dropped (the illegal-harmless quadrant carries such words in 34 of 60 rows), so you can see how much of the signal is the words. If the number barely moves, the signal is not lexical; if it collapses, say so.

In [ ]:
import json, pandas as pd
rows = []
for cond in [RUN, f"{RUN}_prompted"]:
    for target, stratum, tag in DESIGNS:
        r = json.load(open(f"data/processed/probeeval_{cond}_{tag}.json"))
        rows.append({"condition": "prompted" if cond.endswith("_prompted") else "bare", "probe": target, "trained inside": stratum, "layer": r["layer"], "C": r["C"],
                     "test acc": r["test_cross_acc"], "ci": tuple(round(x, 2) for x in r["test_cross_acc_ci95"]), "null p95": r["perm_null_test_cross_acc"]["p95"], "p": r["p_value_vs_null"],
                     "cos(dL,dH)": r["factorial"]["cos_dlegal_dharm"], "dL→legal": r["factorial"]["dlegal_auroc_on_test_topics"], "dL⊥H→legal": r["factorial"]["dlegal_residualised_auroc_on_test_topics"]})
display(pd.DataFrame(rows).round(3))
print("\n--- main design with legality-vocabulary rows dropped ---")
!python scripts/probe_eval.py --run $RUN --target legal --train-stratum harmful --tag L_h2nh_nocue --drop-cue-rows --val-topics $VAL_TOPICS --test-topics $TEST_TOPICS

### 9 · Just ask the model
The cheapest baseline and the one Nanda names first. Shows every sentence to the model and asks "Is this illegal? Yes or No", then "Is this harmful?", and scores the answers, including on the hard corners only. If asking beats the probe, say so; the probe would then be a monitoring convenience, not a discovery. The refusal rate per quadrant is printed too.

In [ ]:
!python scripts/ask_model.py --run $RUN --label legal --model $MODEL
!python scripts/ask_model.py --run $RUN --label harmful --model $MODEL

### 10 · Recompute the headline by hand, and read the probe's mistakes
Rebuilds the main design's test number from the saved activations and the saved topic split, line by line, so you can follow it; then prints the held-out sentences the probe got wrong. Are they the borderline rows? The ones with legality words? Write a paragraph in `journal/verification-log.md`.

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
r = json.load(open(f"data/processed/probeeval_{RUN}_L_h2nh.json")); L, C = r["layer"], r["C"]
z = np.load(f"data/processed/acts_{RUN}.npz"); acts = z["acts"].astype(np.float32); cols = {k[4:]: z[k] for k in z.files if k.startswith("col_")}
df = pd.DataFrame({k: v for k, v in cols.items()}); df["legal"] = df.legal.astype(int); df["harmful"] = df.harmful.astype(int)
if "exclude" in df: keep = df.exclude.astype(int) == 0; df, acts = df[keep].reset_index(drop=True), acts[keep.to_numpy()]
train = df.topic.isin(r["split"]["train_topics"]) & (df.harmful == 1)          # legality probe trained inside the harmful stratum
test  = df.topic.isin(r["split"]["test_topics"])  & (df.harmful == 0)          # tested on the harmless stratum, unseen topics
clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000)).fit(acts[train.to_numpy(), L], df.legal[train])
pred = clf.predict(acts[test.to_numpy(), L]); truth = df.legal[test].to_numpy()
print(f"recomputed cross-stratum test accuracy at layer {L}, C={C}: {(pred == truth).mean():.3f}   (script reported {r['test_cross_acc']:.3f})")
wrong = df[test][pred != truth]
print(f"\n{len(wrong)} mistakes on held-out harmless-stratum sentences:")
for _, w in wrong.iterrows(): print(f"  [{w.quadrant:17s} borderline={w.borderline}] {w.text}")

### 11 · Optional stretch: does the legality direction *do* anything?
Adds the factorial legality direction (and the version with harm removed) to the model's state while it answers "Is this illegal?" for the hard-corner sentences, and measures the shift in the Yes-minus-No logit, not just word flips; invalid answers are counted separately. Strength is swept as a fraction of the typical activation norm at that layer and compared with five random directions of the same norm. A shift that beats the random directions, in the predicted sign, at more than one strength, is an intervention result. Only if hours allow.

In [ ]:
!python scripts/steer_eval.py --run $RUN --dirs data/processed/factorial_dirs_{RUN}_L_h2nh.npz --model $MODEL
import json; r = json.load(open(f"data/processed/steereval_{RUN}.json"))
import pandas as pd; display(pd.DataFrame(r["conditions"]).round(3))

### 12 · Save your notes to GitHub
Commits `journal/` and the edited `data/scenarios.csv` (your hand-check is part of the record). Results are already on Drive.

In [ ]:
!git config user.email "mherje@live.com" && git config user.name "Martin Herje"
!rm -f journal && cp -r "$DRIVE_DIR/journal" journal && git add journal data/scenarios.csv && (git commit -qm "journal + hand-checked scenarios: Colab session" || true) && git -c "$MATS12_GIT_AUTH" push -q origin main && echo pushed
!rm -rf journal && ln -s "$DRIVE_DIR/journal" journal